In [ ]:
%load_ext autoreload
%autoreload 2

# 1. DQN Agent

In [ ]:
import numpy as np
import torch

from briscola import Briscola
from agents.dqn_agent import DQN_Agent, state_to_tensor

In [ ]:
N_EPISODES = 10_000
LOG_INTERVAL = 500
MODEL = "models/150000_dqn.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [ ]:
env = Briscola()
pretrained = True
if pretrained:
    save = torch.load(MODEL, map_location = device)
    agent = DQN_Agent(env, device, save)
    print(f"Agent loaded from checkpoint: {MODEL}")
else:
    agent = DQN_Agent(env, device)
    print("Agent created from scratch.")

In [ ]:
episode_rewards = [] # Hopefully higher as episodes increase
wins = []
initial_epsilon = agent.epsilon

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0
    
    state, _ = env.reset()
    
    while not done:
        action = agent.select_action(state).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_mask = next_state["hand"]
        done = terminated or truncated

        ep_reward += reward
        
        # Convert to tensors to store in experience buffer
        s_tensor = state_to_tensor(state).to(agent.device)
        r_tensor = torch.tensor([reward], dtype = torch.float32, device = agent.device)
        a_tensor = torch.tensor([[action]], dtype = torch.long, device = agent.device)

        if terminated:
            ns_tensor = None
            nm_tensor = None
        else:
            ns_tensor = state_to_tensor(next_state).to(agent.device)
            nm_tensor = torch.tensor(next_mask, dtype = torch.float32, device = agent.device).unsqueeze(0)
        
        # Buffer update
        agent.buffer.push(s_tensor, a_tensor, ns_tensor, r_tensor, nm_tensor)
        
        state = next_state
        
        # Execute a learning step
        agent.learn()

    agent.epsilon = max(agent.eps_end, initial_epsilon - (i / N_EPISODES) * (initial_epsilon - agent.eps_end))
    
    episode_rewards.append(ep_reward)
    if env.player_score > 60:
        wins.append(1)
    else:
        wins.append(0)

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = np.mean(wins) * 100
        recent_wins = np.mean(wins[-LOG_INTERVAL:]) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Win rate (last {LOG_INTERVAL}): {recent_wins:.1f}% | "
            f"Epsilon: {agent.epsilon:.4f}"
        )

torch.save(agent.policy_net.state_dict(), "models/NAMEHOLDER_dqn.pt")
print("Saved.")

In [ ]:
env   = Briscola()
save = torch.load("models/150000_dqn.pt", map_location = device)
agent = DQN_Agent(
    env=env,
    device=device,
    savefile=save,
    epsilon_start=0.0,   # no exploration during eval
    epsilon_end=0.0,
)
agent.policy_net.eval()

# ── Trackers ─────────────────────────────────────────────────
results        = []   # "win" / "loss" / "tie"
score_diffs    = []   # player_score - opponent_score
episode_rewards = []
points_in_outcome = {"win": [], "loss": [], "tie": []}

for ep in range(N_EPISODES):
    state, info = env.reset()
    ep_reward   = 0.0
    done        = False

    while not done:
        action = agent.select_action(state).item()
        state, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        done = terminated or truncated

    episode_rewards.append(ep_reward)

    if env.player_score > 60:
        outcome = "win"
    elif env.player_score < 60:
        outcome = "loss"
    else:
        outcome = "tie"

    results.append(outcome)
    points_in_outcome[outcome].append(env.player_score)

# ── Report ────────────────────────────────────────────────────
wins   = results.count("win")
losses = results.count("loss")
ties   = results.count("tie")

print(f"\n{'='*52}")
print(f"  Evaluation over {N_EPISODES} episodes")
print(f"{'='*52}")
print(f"  Win rate:   {wins/N_EPISODES*100:5.1f}%  ({wins})")
print(f"  Loss rate:  {losses/N_EPISODES*100:5.1f}%  ({losses})")
print(f"  Tie rate:   {ties/N_EPISODES*100:5.1f}%  ({ties})")
print(f"{'─'*52}")
print(f"  Avg shaped reward per episode:      {np.mean(episode_rewards):+.2f}")
print(f"{'─'*52}")
for outcome in ("win", "loss", "tie"):
    pts = points_in_outcome[outcome]
    if pts:
        print(f"  Avg agent points in {outcome}s:         {np.mean(pts):.1f}")
print(f"{'='*52}\n")

# 2. PPO Agent

In [1]:
import numpy as np
import torch

from briscola import Briscola
from agents.ppo_agent import PPO_Agent, state_to_tensor

In [2]:
N_EPISODES = 1_000_000
LOG_INTERVAL = 500
ROLLOUT_STEPS = 2048
MODEL = "..."

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [3]:
env = Briscola()
pretrained = False
if pretrained:
    save = torch.load(MODEL, map_location = device)
    agent = PPO_Agent(env, device, save)
    print(f"Agent loaded from checkpoint: {MODEL}")
else:
    agent = PPO_Agent(env, device)
    print("Agent created from scratch.")

Agent created from scratch.


In [4]:
episode_rewards = [] # Hopefully higher as episodes increase
wins = []
steps = 0

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0

    state, _ = env.reset()

    while not done:
        action, mask, log_prob, entropy, value = agent.select_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        s_tensor = state_to_tensor(state).to(agent.device)

        agent.buffer.push(s_tensor, action, mask, log_prob, reward, value, float(done))

        ep_reward += reward
        state = next_state
        steps += 1

        # ── PPO update: once per rollout, not every step ──
        if steps % ROLLOUT_STEPS == 0:
            # Bootstrap value of the last state
            if not done:
                with torch.no_grad():
                    s_t = state_to_tensor(state).to(device)
                    _, last_value = agent.policy_net(s_t)
                    last_value = last_value.item()
            else:
                last_value = 0.0

            agent.learn(last_value)

    episode_rewards.append(ep_reward)
    if env.player_score > 60:
        wins.append(1)
    else:
        wins.append(0)

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = np.mean(wins) * 100
        recent_wins = np.mean(wins[-LOG_INTERVAL:]) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Win rate (last {LOG_INTERVAL}): {recent_wins:.1f}% | "
        )

torch.save(agent.policy_net.state_dict(), "models/NAMEHOLDER_ppo.pt")
print("Saved.")

Episode    500/1000000 | Mean reward (last 500): +10.80 | Win rate: 51.4% | Win rate (last 500): 51.4% | 
Episode   1000/1000000 | Mean reward (last 500): -12.95 | Win rate: 48.8% | Win rate (last 500): 46.2% | 
Episode   1500/1000000 | Mean reward (last 500): -0.68 | Win rate: 49.0% | Win rate (last 500): 49.4% | 
Episode   2000/1000000 | Mean reward (last 500): +10.93 | Win rate: 49.6% | Win rate (last 500): 51.4% | 
Episode   2500/1000000 | Mean reward (last 500): +11.20 | Win rate: 49.9% | Win rate (last 500): 51.2% | 
Episode   3000/1000000 | Mean reward (last 500): +18.23 | Win rate: 50.4% | Win rate (last 500): 52.8% | 
Episode   3500/1000000 | Mean reward (last 500): +36.14 | Win rate: 51.3% | Win rate (last 500): 57.0% | 
Episode   4000/1000000 | Mean reward (last 500): +38.35 | Win rate: 52.0% | Win rate (last 500): 56.8% | 
Episode   4500/1000000 | Mean reward (last 500): +36.84 | Win rate: 52.6% | Win rate (last 500): 57.2% | 
Episode   5000/1000000 | Mean reward (last 500)